An_Attention-Based_Fusion_for_Handcrafted_and_Deep_Feature_to_Improve_Optical_and_SAR_Image_Matchingを元にした提案手法

1つの画像からDL特徴量とハンドクラフト特徴量をAFF機構を用いて融合し分類する

In [1]:
import torch
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))


cpu


In [3]:
#ハンドクラフト特徴量
import hida_fast
import keijo_fast
import size_module_fast
import pandas as pd
#判別フェーズ
#特徴量抽出
data = "26_0413_moreclean"
pre = "/home/hiromu/energy/"
hida_tappleA = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"{pre}/data/{data}",subfolder="A",method="45rotate",n=9,T=0.4)
result_hidaA = hida_tappleA.run_all()
hida_tappleB = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"{pre}/data/{data}",subfolder="B",method="45rotate",n=9,T=0.4)
result_hidaB = hida_tappleB.run_all()
hida_tappleC = hida_fast.Hida_folder_jikuari_img_xxxx(base_dir=f"{pre}/data/{data}",subfolder="C",method="45rotate",n=9,T=0.4)
result_hidaC = hida_tappleC.run_all()
dfA = pd.DataFrame(result_hidaA, columns=["filename", "R"])
dfB = pd.DataFrame(result_hidaB, columns=["filename", "R"])
dfC = pd.DataFrame(result_hidaC, columns=["filename", "R"])
dfA ["Label"] = "0"
dfB ["Label"] = "1"
dfC ["Label"] = "2"
result_hida = pd.concat([dfA, dfB, dfC], axis=0,ignore_index=True)
size_tappleA = size_module_fast.Size_folder_taikakusen(base_dir=f"{pre}/data/{data}",subfolder="A")
result_sizeA = size_tappleA.run()
size_tappleB = size_module_fast.Size_folder_taikakusen(base_dir=f"{pre}/data/{data}",subfolder="B")
result_sizeB = size_tappleB.run()
size_tappleC = size_module_fast.Size_folder_taikakusen(base_dir=f"{pre}/data/{data}",subfolder="C")
result_sizeC = size_tappleC.run()
dfA = pd.DataFrame(result_sizeA, columns=["filename", "size_count"])                                                    
dfB = pd.DataFrame(result_sizeB, columns=["filename", "size_count"])
dfC = pd.DataFrame(result_sizeC, columns=["filename", "size_count"])
result_size = pd.concat([dfA, dfB, dfC], axis=0,ignore_index=True)
import os
import glob
import cv2
import numpy as np
import pandas as pd
# from scipy import stats # 等級判別には不要なので削除

# --- 1. Hu Moments (案4) の分析クラス (変更なし) ---

class KeijoAnalyzer_HuMoments_Flusser:
    """
    シイタケのマスク画像から形状特徴（Hu Moments + Flusserの8番目）を分析する
    """
    def __init__(self):
        pass

    def analyze_hu_moments(self, img_path):
        """
        提案4：7つのHu MomentsとFlusserの8番目の不変量（h7）を計算し、Log変換して返す
        """
        mask = cv2.imread(img_path)
        if mask is None:
            print(f"⚠️ 読み込み失敗: {img_path}")
            return None

        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"⚠️ 輪郭なし: {img_path}")
            return None

        max_contour = max(contours, key=cv2.contourArea)
        
        if cv2.contourArea(max_contour) < 10:
             print(f"⚠️ 面積が小さすぎる: {img_path}")
             return None

        # 輪郭からモーメントを計算
        M = cv2.moments(max_contour)
        
        if M["m00"] == 0:
            return None
            
        # Hu Moments (phi_1〜phi_7) を計算
        hu = cv2.HuMoments(M)
        
        # 1. 標準の7つの不変量 (h0〜h6, Log変換)
        # sgn(hu) * log10(|hu|) で、サインを保持しつつスケールを調整
        log_hu_standard = -np.sign(hu) * np.log10(np.abs(hu))
        
        # 2. Flusserの8番目の不変量 (h7) を追加
        # $\phi_7$ (hu[6]) の絶対値を取り、Log変換することで反射不変とする
        phi7_abs = np.abs(hu[6])
        # $\phi_7$は通常非常に小さいため、Log変換でスケールを調整
        log_hu_8th = -np.log10(phi7_abs)
        
        # 3. 8つの不変量を結合して返す (h0〜h7)
        final_hu_vector = np.append(log_hu_standard, log_hu_8th)

        return final_hu_vector.flatten()


class Keijo_folder_HuMoments(KeijoAnalyzer_HuMoments_Flusser):
    """
    指定されたフォルダ内の全画像のHu Moments (h0〜h7)を計算し、リストに保存する (CSV保存機能は削除/変更)
    """
    def __init__(self, base_dir, category, mask_dir="mask", subfolder=None):
        super().__init__()
        # subfolderがNoneの場合はcategoryを使用
        _subfolder = subfolder if subfolder is not None else category
        self.folder_path = os.path.join(base_dir, mask_dir, _subfolder)
        self.category = category # カテゴリ情報を追加
        self.results_list = []

    def run(self):
        image_paths = glob.glob(os.path.join(self.folder_path, "*.jpg")) + \
                      glob.glob(os.path.join(self.folder_path, "*.png"))
        
        print(f"📂 フォルダ({self.category}, Hu Moments + h7): {self.folder_path} に画像 {len(image_paths)} 枚")
        
        self.results_list = []
        column_names = ['filename', 'category'] + [f'h{i}' for i in range(8)]

        for img_path in image_paths:
            file_name = os.path.basename(img_path)
            
            # Hu Moments (h0〜h7) を計算
            hu_vector = self.analyze_hu_moments(img_path)
            
            if hu_vector is not None:
                # ファイル名、カテゴリ、8つのモーメントを結合してリストに追加
                row = [file_name, self.category] + list(hu_vector)
                self.results_list.append(row)
            else:
                # 失敗した場合、NaNで埋める
                self.results_list.append([file_name, self.category] + [np.nan] * 8) 
                
        # 結果をDataFrameとして返す
        df = pd.DataFrame(self.results_list, columns=column_names)
        # 特徴量計算に失敗した行を除外
        return df.dropna(subset=[f'h{i}' for i in range(8)]) 


# --- 2. 等級判別用データ準備関数 ---

def prepare_data_for_classification(base_dir, categories):
    """
    指定されたカテゴリの全画像についてHu Moments特徴量（h0〜h7）を計算し、
    一つのDataFrameに結合して返す。
    """
    all_dfs = []
    
    print("--- 等級判別用データ準備 (Hu Moments + h7) ---")
    
    for category in categories:
        # Keijo_folder_HuMoments クラスを使用して特徴量を計算
        analyzer = Keijo_folder_HuMoments(
            base_dir=base_dir,
            category=category,
            # mask_dir="mask", # デフォルト値
            subfolder=category # カテゴリ名がサブフォルダ名と一致
        )
        
        df_category = analyzer.run()
        all_dfs.append(df_category)
        print(f"✅ カテゴリ {category} の特徴量 {len(df_category)} 件を抽出完了。")

    # 全カテゴリのデータを結合
    combined_df = pd.concat(all_dfs, ignore_index=True)

    print(f"\n✨ 全カテゴリのHu Moments特徴量抽出・結合が完了しました。")
    print(f"最終データ件数: {len(combined_df)} 件 (特徴量: h0〜h7)")
    
    return combined_df # 全ての情報を一つのDataFrameとして返す


# --- 3. メイン実行ブロック ---

if __name__ == "__main__":
    
    # --- 設定: 入力元 ---
    # ユーザーの既存のコードから /home/data/{data} の形式を想定
    # 例としてダミーのデータディレクトリを設定します
    # INPUT_DATA_DIR = "/home/data/1124_keijotest"
    
    # ユーザーの既存のコード（Keijo_fast）に合わせて、データディレクトリを変数で設定
    # data = "1124_keijotest" # 判別したいデータセット名に置き換えてください
    INPUT_DATA_DIR = f"/{pre}/data/{data}"
    
    CATEGORIES = ["A", "B", "C"]
       
    # Hu Moments (h0〜h7) の特徴量を結合したDataFrameを作成
    result_hu_moments_df = prepare_data_for_classification(
        base_dir=INPUT_DATA_DIR,
        categories=CATEGORIES
    )
    
    # 抽出結果の確認
    print("\n--- 抽出された特徴量データ（先頭5行） ---")
    print(result_hu_moments_df.head())
    
    # ユーザーの既存の出力形式（result_keijo）に合わせた出力例：
    # Hu Momentsの特徴量すべてをそのまま使用するのが最も簡単で強力ですが、
    # もしMSEと同様に単一の特徴量だけを抽出したければ、以下の処理を追加できます。
    # 例：h7だけを抽出
    # result_keijo = result_hu_moments_df[['filename', 'h7']].rename(columns={'h7': 'HU_H7'})
    
    # Hu Momentsの特徴量すべてを含むDataFrameを等級判別用として利用します。
    result_keijo = result_hu_moments_df
    
    print("\n--- 等級判別用の最終データフレーム ---")
    print(result_keijo.head())
    
    # この result_keijo を使って、機械学習モデルで等級判別を行うことができます。
    df_merged = pd.merge(result_keijo, result_size, on="filename")
    df_merged = pd.merge(df_merged, result_hida, on="filename")
    df_merged.to_csv(f"{pre}/data/{data}/feature_huall.csv", index=False)


探索対象フォルダ: /home/hiromu/energy//data/26_0413_moreclean/mask/A
📂 356 枚の画像をマルチプロセスで処理します。


処理中 (A): 100%|██████████| 356/356 [00:01<00:00, 329.28it/s]

探索対象フォルダ: /home/hiromu/energy//data/26_0413_moreclean/mask/B
📂 265 枚の画像をマルチプロセスで処理します。



処理中 (B): 100%|██████████| 265/265 [00:01<00:00, 261.32it/s]

探索対象フォルダ: /home/hiromu/energy//data/26_0413_moreclean/mask/C
📂 215 枚の画像をマルチプロセスで処理します。



処理中 (C): 100%|██████████| 215/215 [00:00<00:00, 257.29it/s]


--- 等級判別用データ準備 (Hu Moments + h7) ---
📂 フォルダ(A, Hu Moments + h7): //home/hiromu/energy//data/26_0413_moreclean/mask/A に画像 356 枚
✅ カテゴリ A の特徴量 356 件を抽出完了。
📂 フォルダ(B, Hu Moments + h7): //home/hiromu/energy//data/26_0413_moreclean/mask/B に画像 265 枚
✅ カテゴリ B の特徴量 265 件を抽出完了。
📂 フォルダ(C, Hu Moments + h7): //home/hiromu/energy//data/26_0413_moreclean/mask/C に画像 215 枚
✅ カテゴリ C の特徴量 215 件を抽出完了。

✨ 全カテゴリのHu Moments特徴量抽出・結合が完了しました。
最終データ件数: 836 件 (特徴量: h0〜h7)

--- 抽出された特徴量データ（先頭5行） ---
             filename category        h0        h1        h2        h3  \
0  IMG_66639_mask.png        A  0.793217  3.268717  5.228053  7.575564   
1  IMG_66517_mask.png        A  0.796533  4.159862  4.806959  7.786477   
2  IMG_66627_mask.png        A  0.792293  3.547033  4.348787  6.304488   
3  IMG_64219_mask.png        A  0.781341  2.740575  4.705317  6.388784   
4   IMG_6445_mask.png        A  0.795178  3.640920  4.778151  7.421268   

          h4        h5         h6         h7  
0 -14.057758 -9.293075  14.23211

↓gakusyuで学習，testで検証

Cross-Attentionのほうが良いのでは？